# AI-Lake + Trino

Trino reads AI-Lake tables via the standard **Iceberg connector** —
no AI-Lake plugin required for tabular queries.

**Prerequisites:** start the engines compose overlay:
```bash
docker compose \
  -f tests/docker/compose-demo.yml \
  --profile engines \
  up -d
```

**What this notebook covers:**
1. Connect to Trino via the Python `trino` driver
2. Discover catalogs, schemas, tables
3. SQL queries: scan, filter, aggregation
4. `embedding` column as `varbinary`
5. Iceberg metadata queries (`$snapshots`, `$manifests`)

Sections 11-13 additionally connect to the **`ailake_native` catalog** (the real `trino-plugin` JAR, built into this image) — real plugin loading, session properties, `EXPLAIN` planning, and real `SELECT` execution end-to-end.

In [ ]:
import os
import trino

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

print(f'Connecting to Trino at {TRINO_HOST}:{TRINO_PORT} ...')

conn = trino.dbapi.connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user='ailake',
    catalog='ailake',
    schema='default',
    request_timeout=60,
)
cur = conn.cursor()

def q(sql):
    """Execute SQL and return rows."""
    cur.execute(sql)
    return cur.fetchall()

def qdf(sql):
    """Execute SQL and return a pandas DataFrame."""
    import pandas as pd
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

print('Connected.')

In [ ]:
# ── Pre-flight check — verify Trino is reachable ─────────────────────────────
import os, socket

TRINO_HOST = os.environ.get('TRINO_HOST', 'trino')
TRINO_PORT = int(os.environ.get('TRINO_PORT', '8080'))

try:
    with socket.create_connection((TRINO_HOST, TRINO_PORT), timeout=5):
        print(f"OK: Trino reachable at {TRINO_HOST}:{TRINO_PORT}")
except OSError:
    raise RuntimeError(
        f"\n\nTrino is not reachable at {TRINO_HOST}:{TRINO_PORT}.\n"
        "Start the engines overlay first:\n\n"
        "  docker compose \\\n"
        "    -f tests/docker/compose-demo.yml \\\n"
        "    --profile engines \\\n"
        "    up -d\n"
    )

## 1. Discover catalogs and tables

In [ ]:
catalogs = q('SHOW CATALOGS')
schemas  = q('SHOW SCHEMAS FROM ailake')
print('Catalogs:', catalogs)
print('Schemas :', schemas)

# Guard: 'default' must be present — if only information_schema is visible,
# Nessie registration hasn't run yet. Rebuild or restart the jupyter container.
schema_names = [r[0] for r in schemas]
if 'default' not in schema_names:
    raise RuntimeError(
        "Schema 'default' missing from ailake catalog.\n"
        "Nessie registration in init_demo.py may have failed at container startup.\n"
        "Restart the jupyter container to re-run init_demo.py:\n\n"
        "  docker compose -f tests/docker/compose-demo.yml restart jupyter\n\n"
        "Then wait ~30 s and re-open this notebook."
    )

print('Tables  :', q('SHOW TABLES FROM ailake.default'))

## 2. Schema inspection

In [ ]:
# embedding is varbinary — standard Parquet FIXED_LEN_BYTE_ARRAY read as bytes
qdf('DESCRIBE ailake.default."table"')

## 3. Basic scan

In [ ]:
count = q('SELECT count(*) FROM ailake.default."table"')[0][0]
print(f'Total rows: {count}')

In [ ]:
qdf("""
    SELECT text,
           length(embedding) AS embedding_bytes
    FROM ailake.default."table"
    LIMIT 5
""")

## 4. Filtered query + aggregation

In [ ]:
df = qdf("""
    SELECT text
    FROM ailake.default."table"
    WHERE text LIKE '%vector search%'
""")
print(f"Rows about 'vector search': {len(df)}")
df.head(3)

In [ ]:
qdf("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_chars,
        avg(length(embedding)) AS avg_embedding_bytes
    FROM ailake.default."table"
""")

## 5. Iceberg metadata — snapshots and manifests

In [ ]:
# Trino Iceberg connector exposes system tables via the $ suffix
qdf('SELECT snapshot_id, committed_at, operation FROM "ailake"."default"."table$snapshots"')

In [ ]:
qdf("""
    SELECT
        file_path,
        record_count,
        file_size_in_bytes
    FROM "ailake"."default"."table$files"
""")

## 6. Table properties — AI-Lake custom metadata via Trino

AI-Lake stores vector configuration in Iceberg table properties.
Trino exposes these via the `$properties` system table.

> `ailake.embedding-model` appears here when the table was written with
> `embedding_model=` — e.g. `ailake.open_table(..., embedding_model="text-embedding-3-small")`.


In [ ]:
props = qdf('SELECT * FROM "ailake"."default"."table$properties"')
# Filter AI-Lake properties
ailake_props = props[props['key'].str.startswith('ailake.')]
print(f'AI-Lake properties in Trino Iceberg catalog:')
ailake_props


## 7. File-level manifest statistics

Trino exposes per-file record counts and sizes from the Iceberg manifest via `$files`.


In [ ]:
sql = (
    "SELECT element_at(split(file_path, '/'), -1) AS filename,"
    " record_count, file_size_in_bytes,"
    " round(cast(file_size_in_bytes AS double) / record_count, 1) AS bytes_per_row"
    ' FROM \"ailake\".\"default\".\"table$files\"'
)
qdf(sql)


## 8. Manifest files

`$manifests` lists the Avro manifest files that compose each Iceberg snapshot.
Trino reads these to discover which Parquet files to scan.


In [ ]:
qdf('SELECT path, length, partition_spec_id, added_data_files_count FROM "ailake"."default"."table$manifests"')


## 9. Embedding model tracking via Trino `$properties`

When a table has `ailake.embedding-model` set, it appears in the Trino
`$properties` system table. Use it to audit which model version was used
and to validate before running `migrate_embeddings()` from the SDK.

In [ ]:
# Filter embedding-model properties specifically
emb_df = qdf(
    'SELECT "key", "value" FROM "ailake"."default"."table$properties"'
    ' WHERE "key" LIKE \'ailake.embedding%\''
)
if len(emb_df) > 0:
    print('Embedding model properties visible in Trino:')
    print(emb_df.to_string(index=False))
else:
    print('(ailake.embedding-model not set — write with embedding_model= to populate)')


## 10. Iceberg v3 partitioned table via Trino

`init_demo.py` registers `partitioned_v3`, `delete_demo`, and `schema_evo` in Nessie
so Trino can discover them alongside the main HNSW table.

These tables demonstrate Phase L–N features observable via standard Trino SQL:
- `partitioned_v3` — format_version=3, `topic_id` partition (identity transform)
- `delete_demo` — equality delete files visible in `$manifests` / `$files`
- `schema_evo` — evolved schema (`source_url` column) visible in `DESCRIBE`


In [ ]:
# partitioned_v3 — Iceberg v3 with topic_id partition
print("=== partitioned_v3: row count and format-version ===")
try:
    cnt = qdf('SELECT COUNT(*) AS total_rows FROM ailake."default".partitioned_v3')
    print(cnt.to_string(index=False))

    # Table DDL includes partition spec
    print("\nTable DDL (includes PARTITIONED BY):")
    ddl_rows = q('SHOW CREATE TABLE ailake."default".partitioned_v3')
    print(ddl_rows[0][0] if ddl_rows else "(none)")

    # format-version and ailake properties
    print("\nICEBERG + ailake properties:")
    props = qdf(
        "SELECT key, value"
        ' FROM ailake.default."partitioned_v3$properties"'
        " WHERE key IN ('format-version', 'ailake.format-version',"
        "               'ailake.vector-dim', 'ailake.vector-metric')"
    )
    print(props.to_string(index=False))

    # Per-partition record counts
    print("\nPartition record counts ($partitions):")
    qdf(
        "SELECT record_count, file_count"
        ' FROM ailake.default."partitioned_v3$partitions"'
        " ORDER BY record_count DESC LIMIT 5"
    )
except Exception as e:
    msg = str(e)
    if "unsupported version 3" in msg:
        print(f"partitioned_v3 skipped (format-version 3 unsupported by this Trino version): {msg}")
    elif "not exist" in msg or "TABLE_NOT_FOUND" in msg:
        print(f"partitioned_v3 not registered in Nessie yet (needs --profile engines + Nessie init): {e}")
    else:
        print(f"partitioned_v3 error: {e}")


In [ ]:
# delete_demo — rows 0-9 pre-deleted at init_demo.py; rows 10-19 deleted later.
print("=== delete_demo: manifest check + visible rows ===")

# 1. Manifest check — reads Nessie metadata only, no data scan
try:
    mdf = qdf(
        'SELECT path, length, added_data_files_count'
        ' FROM ailake."default"."delete_demo$manifests"'
        ' ORDER BY path'
    )
    eq_del_manifests = mdf[mdf["path"].str.contains("eq-del")]
    print(f"  Total manifests     : {len(mdf)}")
    print(f"  Eq-delete manifests : {len(eq_del_manifests)} (committed to Nessie)")
    if len(eq_del_manifests) > 0:
        print("  PASS — equality delete files are committed in Iceberg metadata.")
    else:
        print("  NOTE — no eq-del manifests found (table may use position deletes).")
except Exception as e:
    print(f"  manifest check error: {e}")

# 2. Row count — requires Trino to merge delete files (MOR scan)
print()
try:
    res = qdf('SELECT COUNT(*) AS visible_rows FROM ailake."default".delete_demo')
    print(res.to_string(index=False))
    print("  (expect 90 — rows 0-9 deleted)")
except Exception as e:
    print(f"  visible_rows error: {e}")
    print("  (NPE on delete file read is a known Trino <450 bug; fixed in Trino 460)")


In [ ]:
# schema_evo — add_column(source_url, string) applied at init; visible in DESCRIBE
print("=== schema_evo: schema after add_column ===")
try:
    schema_df = qdf('DESCRIBE ailake."default".schema_evo')
    print(schema_df.to_string(index=False))
    print()
    # source_url column should be present (null in existing rows)
    qdf('SELECT text, source_url FROM ailake."default".schema_evo LIMIT 3')
except Exception as e:
    print(f"schema_evo not registered in Nessie yet: {e}")


## 11. ailake Trino plugin — `ailake_native` catalog (real connector)

Sections 1-10 above used Trino's **stock Iceberg connector** — no AI-Lake code
runs inside Trino. This section connects to a *second* catalog,
`ailake_native`, backed by `io.ailake.trino.VectorScanConnectorFactory`
(`trino-plugin/`, built into this image's `Dockerfile.trino`, pinned to
**Trino 430** — see that file's header for why). It exposes
`ailake_native.default.{search, search_full, search_multimodal, ingest}` as
native SQL, backed directly by `libailake_jni.so` — no Iceberg connector
involved for this catalog at all.

In [ ]:
native_conn = trino.dbapi.connect(
    host=TRINO_HOST, port=TRINO_PORT, user='ailake',
    catalog='ailake_native', schema='default', request_timeout=60,
)
ncur = native_conn.cursor()

def nq(sql):
    ncur.execute(sql)
    return ncur.fetchall()

print('Schemas:', nq('SHOW SCHEMAS'))
print('Tables :', nq('SHOW TABLES'))

## 12. Session properties + query plan (real — `SET SESSION` and `EXPLAIN` both execute)

The query vector and `top_k` are passed as **session properties**
(`ailake_native.query_vector`, `.top_k`), not SQL arguments — there's no
`vector_search()` function (the closing note in earlier revisions of this
notebook was aspirational; the real design uses session properties, see
`VectorScanConnector.getSessionProperties()`). `EXPLAIN` confirms the plan
resolves the right physical table (`tableUri`/`tableName`) without running
any task — this part of the pipeline (planning) works end-to-end.

In [ ]:
import pathlib, json

with open(str(pathlib.Path('/data/demo_query.json'))) as f:
    demo_q = json.load(f)
query_csv = ','.join(str(x) for x in demo_q['query_vector'])

nq(f"SET SESSION ailake_native.query_vector = '{query_csv}'")
nq("SET SESSION ailake_native.top_k = 5")
print('Session properties set.')

plan = nq('EXPLAIN SELECT row_id, distance FROM search')
print(plan[0][0])

## 13. `SELECT` execution — two real bugs found and fixed

Planning works (§12); running the query now works too. `SELECT` against
`search`/`search_full`/`search_multimodal` previously failed with a
`NullPointerException` reconstructing `VectorScanTableHandle` — Trino
serializes the table handle to JSON as part of its internal
`TaskUpdateRequest` (coordinator → worker HTTP call, exercised even in
single-node mode), and `tableUri` came back null on the receiving end despite
a `@JsonCreator`/`@JsonProperty`-annotated Kotlin data class — even though the
*exact same* handle rendered correctly with `tableUri` populated in the
`EXPLAIN` plan text above. Root cause: a bare `@JsonProperty` on a Kotlin
primary-constructor `val` only reaches Jackson's *parameter* use-site, not a
getter — and Trino's `ObjectMapperProvider` disables
`MapperFeature.AUTO_DETECT_GETTERS`/`AUTO_DETECT_FIELDS` globally, so every
field silently serialized as absent. Fixed by pairing every
`@param:JsonProperty(...)` with an explicit `@get:JsonProperty(...)` on every
`ConnectorTableHandle`/`ConnectorSplit`/`ColumnHandle` data class.

Fixing that surfaced a second bug: `VectorScanTransactionHandle` is a Kotlin
`object` (stateless singleton) — a private synthetic no-arg constructor that
Trino's internal (non-Kotlin-aware) Jackson mapper couldn't reflect into
(`IllegalAccessException`). Fixed with a `@JsonCreator @JvmStatic` factory
method that returns the singleton without touching the constructor.

Both bugs — plus two *other* real bugs found and fixed getting this far, both
confirmed against this live Trino 430 server, none previously covered by any
test in this repo — are documented in `CHANGELOG.md`:
- `slf4j-api` was `compileOnly` in `build.gradle.kts`; Trino's per-plugin
  classloader does not expose it, so the connector failed to construct at all
  (`NoClassDefFoundError`) until it was bundled into the shadowJar.
- `session.getProperty("top_k", Int::class.java)` — Kotlin's `Int::class.java`
  resolves to the *primitive* `int.class`, not the boxed `java.lang.Integer`
  `PropertyMetadata.integerProperty` registers; every query calling `SET
  SESSION`/executing failed with `INVALID_SESSION_PROPERTY` until switched to
  `Int::class.javaObjectType` (same fix applied to `hybrid_weight`/`Double`).

None of this was caught by review — all four bugs were found by standing up
the real plugin against a live Trino 430 server and running real queries.


In [ ]:
rows = nq('SELECT row_id, distance, file_path FROM search')
for r in rows:
    print(r)


## Key takeaway

Trino queries AI-Lake tables through the **standard Iceberg connector** —
the Parquet files and Iceberg metadata are fully compatible. The `embedding`
column appears as `varbinary`.

Sections 11-13 deployed the real `trino-plugin` JAR (`ailake_native` catalog,
Trino 430) and now work end-to-end: connects, registers
`search`/`search_full`/`search_multimodal`/`ingest`, accepts session
properties, plans correctly, and **executes** real `SELECT` queries (§13).
Four real bugs (slf4j classloading, Kotlin session-property boxing, two
Jackson serialization bugs in Trino's internal task codec) were found and
fixed getting here — none caught by unit tests, all confirmed against a live
Trino 430 server. Vector search also works today via the AI-Lake SDK
(`ailake.search()` — `01_ailake_demo.ipynb`) and the DuckDB/Spark plugins
(`02_duckdb.ipynb`, `03_spark.ipynb`).